# Data Pipeline

## Imports

In [1]:
import os
import json
from pathlib import Path
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from fastembed import SparseTextEmbedding
from openai import AzureOpenAI
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv

c:\Users\abarhouche\Desktop\GenAI-Pinnacle-Program\GenAI-Pinnacle-Program\25_Capstone\dev\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


## Load Environment Variables

In [2]:
load_dotenv()

True

## Helper Functions

In [3]:
def get_reference(text, deployment = "gpt-5.4", api_version = "2024-12-01-preview"):

    sys_prompt = """You are an AI assistant that extracts the bibliographic reference of a scientific paper itself by analyzing its first page. 
Your task is to identify the paper’s own citation details and return them in JSON format with the following fields:

{
  "reference": {
    "title": "string or N/A",
    "first_author": "string or N/A",
    "year": "string or N/A",
    "publication": "string or N/A"
  }
}

Rules:
- Always output valid JSON only, with no extra commentary.
- "title" is the title of the paper.
- "first_author" is the first listed author of the paper.
- "year" is the publication year of the paper.
- "publication" is the journal, conference, or book where the paper was published.
- If any field cannot be found on the first page, set its value to "N/A".
- Do not attempt to extract references cited by the paper; only extract the reference of the paper itself.
"""

    client = AzureOpenAI(
        api_version=api_version,
        azure_endpoint=os.environ["AZURE_ENDPOINT"],
        api_key=os.environ["AZURE_API_KEY"],
    )

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": text,
            }
        ],
        max_completion_tokens=2048,
        model=deployment
    )

    return json.loads(response.choices[0].message.content)

def format_reference(reference):
    _r = reference["reference"]
    return f"{_r['title']}, {_r['first_author']}, {_r['year']}"

def create_chunks(documents, references):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2500,    # Max characters per chunk, in average an english word is 5 characters
        chunk_overlap=0,  # Characters to repeat between chunks to keep context
        length_function=len,
        is_separator_regex=False,
    )

    chunks = []

    for doc, ref in zip(documents, references):
        ref_string = format_reference(ref)
        for page in doc:
            page_ref_string = ref_string + f", page {page['metadata']['page_number']}."
            _chunks = text_splitter.split_text(page["text"])
            _chunks = [f"Source: {page_ref_string}\n---\n{c}" for c in _chunks]
            chunks.extend(_chunks)
    
    return chunks

In [4]:
def prepare_pdf_chunks(file_dir):
    
    documents = []
    references = []
    
    for file in Path(file_dir).glob("*.pdf"):
        print(f"Parsing file: {file}")
        pages = pymupdf4llm.to_markdown(file, force_ocr=True, page_chunks=True)
        documents.append(pages)

    for document in documents:
        reference = get_reference(document[0]["text"])
        references.append(reference)
    
    chunks = create_chunks(documents, references)

    return chunks


In [5]:
file_dir = "../test_data"

chunks = prepare_pdf_chunks(file_dir)

Parsing file: ..\test_data\attention_paper.pdf


## testing 

### markitdown

In [ ]:
from pathlib import Path
from markitdown import MarkItDown

def prepare_pdf_chunks(file_dir):
    # Initialize the MarkItDown converter
    md = MarkItDown()
    
    documents = []
    references = []
    
    for file in Path(file_dir).glob("*.pdf"):
        print(f"Parsing file: {file}")
        
        # Convert PDF to Markdown text content
        # .convert() returns a DocumentConverterResult object
        result = md.convert(str(file))
        
        # To match your original structure (list of dicts), 
        # we wrap the result in a single-element list.
        page_data = [{"text": result.text_content}]
        documents.append(page_data)

    for document in documents:
        # document[0]["text"] now contains the full markdown of the PDF
        reference = get_reference(document[0]["text"])
        references.append(reference)
    
    chunks = create_chunks(documents, references)

    return chunks


In [ ]:
chunks = prepare_pdf_chunks(file_dir)

### docling

In [ ]:
from pathlib import Path
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions

# 1. Configure the pipeline options
pipeline_options = PdfPipelineOptions()
pipeline_options.do_formula_enrichment = True  # Enable LaTeX math extraction

def prepare_pdf_chunks(file_dir):
    # 2. Assign the options specifically to the PDF format
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
            }
            )
    # converter = DocumentConverter()
    documents = []
    references = []
    
    for file in Path(file_dir).glob("*.pdf"):
        print(f"Parsing file: {file}")
        # convert() returns a ConversionResult
        result = converter.convert(str(file))
        doc_obj = result.document # This is a DoclingDocument object
        
        # We transform the Docling structure into a list of "pages" 
        # to match your original processing logic.
        pages = []
        for page_no, page in doc_obj.pages.items():
            # Docling allows exporting specific page content to markdown
            page_text = doc_obj.export_to_markdown(page_no=page_no)
            pages.append({
                "text": page_text,
                "metadata": {"page_number": page_no}
            })
        
        documents.append(pages)

    for pages in documents:
        # Pass the text from the first page to your extraction logic
        reference = get_reference(pages[0]["text"])
        references.append(reference)
    
    return create_chunks(documents, references)


In [ ]:
chunks = prepare_pdf_chunks(file_dir)

In [ ]:
chunks

### nougat

In [11]:
import sys
import subprocess
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_active_nougat_executable() -> Path:
    """Dynamically finds the nougat.exe matching the active Python runtime environment."""
    # sys.executable points directly to the active interpreter (e.g., .../.venv/Scripts/python.exe)
    active_python_dir = Path(sys.executable).parent
    nougat_exe = active_python_dir / "nougat.exe"
    
    if not nougat_exe.exists():
        # Fallback diagnostic check for Windows path variations
        nougat_exe_alternative = active_python_dir / "Scripts" / "nougat.exe"
        if nougat_exe_alternative.exists():
            return nougat_exe_alternative
            
        raise FileNotFoundError(
            f"\n\n[NOUGAT NOT FOUND ERROR]\n"
            f"The Nougat executable is missing from your active environment.\n"
            f"Expected Location: {nougat_exe}\n"
            f"Active Python Path: {sys.executable}\n"
            f"👉 Fix this by running: pip install nougat-ocr\n"
        )
    return nougat_exe

def run_nougat_ocr(file_path: Path, output_dir: Path) -> Path:
    """Executes Nougat CLI using dynamic absolute paths."""
    abs_file = Path(file_path).resolve()
    abs_out = Path(output_dir).resolve()
    
    # Dynamically find the binary path based on your active runtime environment
    nougat_executable = get_active_nougat_executable()
    
    if not abs_file.exists():
        raise FileNotFoundError(f"Target PDF file does not exist at: {abs_file}")
        
    abs_out.mkdir(parents=True, exist_ok=True)

    # Formal execution list array for shell=False security
    command = [
        str(nougat_executable),
        str(abs_file),
        "--out", str(abs_out),
        "--markdown",
        "--no-skipping",
        "--full-precision"
    ]
    
    print(f"Running dynamic binary path: {nougat_executable}")
    result = subprocess.run(command, shell=False, capture_output=True, text=True)
    
    if result.returncode != 0:
        print("\n=== NOUGAT CORE ERROR LOG ===")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
        print("==============================\n")
        raise subprocess.CalledProcessError(result.returncode, command)
        
    return abs_out / f"{abs_file.stem}.mmd"

# Use the rest of your `parse_mmd_pages`, `create_chunks`, and `prepare_pdf_chunks` functions unchanged.


In [12]:
import subprocess
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
import shutil

import subprocess
from pathlib import Path

import subprocess
from pathlib import Path


def parse_mmd_pages(mmd_path: Path) -> list[dict]:
    """Parses Nougat's .mmd file and splits it into logical pages using the [bkmk] or \x0c delimiters."""
    with open(mmd_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Nougat injects form feeds (\x0c) or specific markdown markers for page breaks
    raw_pages = content.split("\x0c") 
    
    pages_data = []
    for idx, page_text in enumerate(raw_pages, start=1):
        if page_text.strip():
            pages_data.append({
                "text": page_text.strip(),
                "metadata": {"page_number": idx}
            })
            
    return pages_data

def create_chunks(documents, references):
    # Expanded chunk size slightly to avoid cutting complex multi-line LaTeX matrices
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=3000,    
        chunk_overlap=300,  # Overlap added to prevent splitting equations across chunks
        length_function=len,
        is_separator_regex=False,
        # Separators optimized for math papers and markdown structure
        separators=["\n\n", "\n", " ", ""] 
    )

    chunks = []

    for doc, ref in zip(documents, references):
        ref_string = format_reference(ref)
        for page in doc:
            page_ref_string = ref_string + f", page {page['metadata']['page_number']}."
            _chunks = text_splitter.split_text(page["text"])
            
            # Formatted clean string prefix for Qdrant payload structuring
            _chunks = [f"Source: {page_ref_string}\n---\n{c}" for c in _chunks]
            chunks.extend(_chunks)
    
    return chunks

def prepare_pdf_chunks(file_dir, temp_output_dir="./nougat_output"):
    documents = []
    references = []
    
    out_path = Path(temp_output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    
    # 1. Process PDFs through Nougat
    for file in Path(file_dir).glob("*.pdf"):
        print(f"Parsing file with Nougat: {file.name}")
        try:
            mmd_file = run_nougat_ocr(file, out_path)
            pages = parse_mmd_pages(mmd_file)
            documents.append(pages)
        except Exception as e:
            print(f"Failed parsing {file.name}: {e}")
            continue

    # 2. Extract references from the parsed content
    for document in documents:
        if document:  # Ensure document is not empty
            reference = get_reference(document[0]["text"])
            references.append(reference)
    
    # 3. Create chunks for Vector DB
    chunks = create_chunks(documents, references)

    return chunks


In [ ]:
file_dir = "../test_data"
chunks = prepare_pdf_chunks(file_dir)

Parsing file with Nougat: attention_paper.pdf
Running dynamic binary path: c:\Users\abarhouche\Desktop\GenAI-Pinnacle-Program\GenAI-Pinnacle-Program\25_Capstone\dev\.venv\Scripts\nougat.exe


In [ ]:
def get_dense_emb(text, model_name = "text-embedding-3-large"):

    client = AzureOpenAI(
        api_version="2024-12-01-preview",
        azure_endpoint=os.environ["AZURE_ENDPOINT"],
        api_key=os.environ["AZURE_API_KEY"]
    )

    response = client.embeddings.create(
        input=[text],
        model=model_name
    )

    return response.data[0].embedding

def get_bm25_emb(text):
 
    model = SparseTextEmbedding(model_name="Qdrant/bm25")
    return list(model.embed([text]))[0]


In [ ]:
points = []

for i, text in enumerate(chunks):
    # 1. Generate Embeddings
    dense_vector = get_dense_emb(text)
    sparse_embedding = get_bm25_emb(text) # Returns a SparseVector object
    
    # 2. Map to Qdrant Point structure
    points.append(
        models.PointStruct(
            id=i,
            vector={
                "dense-vector": dense_vector,
                "sparse-bm25": models.SparseVector(
                    indices=sparse_embedding.indices.tolist(), 
                    values=sparse_embedding.values.tolist()
                )
            },
            payload={
                "text": text,
                # Add any other metadata here
            }
        )
    )


In [ ]:
# Initialize client (assuming collection is already created as shown previously)
client = QdrantClient(
    url=os.environ["QDRANT_ENDPOINT"],
    api_key=os.environ["QDRANT_API_KEY"]
)

# Assuming 'points' is your list of PointStructs
batch_size = 50 

for i in range(0, len(points), batch_size):
    batch = points[i : i + batch_size]
    client.upsert(
        collection_name="rag_database",
        points=batch
    )
    print(f"Uploaded batch {i // batch_size + 1}")


In [ ]:
query_text = "What can you tell me about Attention?"

# 1. Generate query embeddings using your functions
dense_query = get_dense_emb(query_text)
sparse_query_obj = get_bm25_emb(query_text)

# 2. Execute Hybrid Search (RRF)
results = client.query_points(
    collection_name="rag_database",
    prefetch=[
        # Dense Search
        models.Prefetch(
            query=dense_query, 
            using="dense-vector", 
            limit=20
        ),
        # Sparse BM25 Search
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_obj.indices.tolist(), 
                values=sparse_query_obj.values.tolist()
            ), 
            using="sparse-bm25", 
            limit=20
        ),
    ],
    # Combine results using Reciprocal Rank Fusion
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=10
)

# Display results
for point in results.points:
    print(f"ID: {point.id}, Score: {point.score}, Text: {point.payload['text'][:100]}...")
